In [1]:
# ================================
# 1. IMPORT LIBRARIES
# ================================
import pandas as pd
import numpy as np
import nltk
import spacy
import re

from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer, WordNetLemmatizer
from nltk import pos_tag
from nltk.util import ngrams

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import StackingClassifier
from sklearn.metrics import classification_report, accuracy_score



In [2]:
# ================================
# 2. DOWNLOAD NLTK DATA
# ================================
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('averaged_perceptron_tagger')


[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\AsusROG1\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\AsusROG1\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\AsusROG1\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     C:\Users\AsusROG1\AppData\Roaming\nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!


True

In [3]:
# ================================
# 3. LOAD DATA
# ================================
df = pd.read_csv(r"E:\12. Tamanna\Goonj\Dataset\Mental20Text20for20and20Classification.csv")  

# Assuming:
# df["text"] = patient speech
# df["status"] = anxiety / stress / bipolar / personality disorder

print(df.head())



                                                text                status
0  "My mind is a never-ending cycle of worry, and...               anxiety
1  Despite the sun shining and birds singing outs...               bipolar
2  I'm drowning in responsibilities, each one dem...                stress
3  "My emotions shift like the wind, leaving me u...  personality disorder
4  I'm trapped in a whirlwind of thoughts, unable...               anxiety


In [4]:
# ================================
# 4. LOWERCASE + CLEANING
# ================================
def clean_text(text):
    text = text.lower()
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    return text

df["clean_text"] = df["text"].apply(clean_text)


In [5]:
# ================================
# 5. TOKENIZATION
# ================================
df["tokens"] = df["clean_text"].apply(word_tokenize)


In [6]:
# ================================
# 6. REMOVE STOPWORDS
# ================================
stop_words = set(stopwords.words("english"))

def remove_stopwords(tokens):
    return [word for word in tokens if word not in stop_words]

df["tokens_no_stop"] = df["tokens"].apply(remove_stopwords)



In [7]:
# ================================
# 7. STEMMING
# ================================
stemmer = PorterStemmer()

def stem_words(tokens):
    return [stemmer.stem(word) for word in tokens]

df["stemmed"] = df["tokens_no_stop"].apply(stem_words)



In [8]:
# ================================
# 8. LEMMATIZATION
# ================================
lemmatizer = WordNetLemmatizer()

def lemmatize_words(tokens):
    return [lemmatizer.lemmatize(word) for word in tokens]

df["lemmatized"] = df["tokens_no_stop"].apply(lemmatize_words)

#

In [9]:
# ================================
# 9. POS TAGGING
# ================================
df["pos_tags"] = df["tokens_no_stop"].apply(pos_tag)

print("Sample POS tagging:")
print(df["pos_tags"].iloc[0])


Sample POS tagging:
[('mind', 'NN'), ('neverending', 'VBG'), ('cycle', 'NN'), ('worry', 'NN'), ('even', 'RB'), ('simplest', 'JJS'), ('tasks', 'NNS'), ('feel', 'VBP'), ('insurmountable', 'JJ'), ('im', 'NN'), ('consumed', 'VBD'), ('fear', 'JJ'), ('doubt', 'NN'), ('every', 'DT'), ('decision', 'NN'), ('feels', 'NNS'), ('like', 'IN'), ('minefield', 'NN'), ('waiting', 'VBG'), ('explode', 'JJ'), ('anxiety', 'NN'), ('grip', 'NN'), ('im', 'NN'), ('powerless', 'NN'), ('break', 'NN'), ('free', 'JJ'), ('relentless', 'NN'), ('hold', 'NN'), ('thoughts', 'NNS'), ('race', 'NN'), ('like', 'IN'), ('runaway', 'NN'), ('train', 'NN'), ('cant', 'NN'), ('seem', 'VBP'), ('find', 'VB'), ('way', 'NN'), ('slow', 'JJ'), ('catch', 'NN'), ('breath', 'NN'), ('im', 'NN'), ('trapped', 'VBD'), ('whirlwind', 'IN'), ('fear', 'JJ'), ('uncertainty', 'NN'), ('cant', 'NN'), ('seem', 'VBP'), ('find', 'VB'), ('way', 'NN'), ('escape', 'IN'), ('heart', 'NN'), ('pounds', 'NNS'), ('chest', 'VBP')]


In [10]:

# ================================
# 10. UNIGRAM & BIGRAM
# ================================
def get_bigrams(tokens):
    return list(ngrams(tokens, 2))

df["bigrams"] = df["tokens_no_stop"].apply(get_bigrams)

print("Sample bigrams:")
print(df["bigrams"].iloc[0])


Sample bigrams:
[('mind', 'neverending'), ('neverending', 'cycle'), ('cycle', 'worry'), ('worry', 'even'), ('even', 'simplest'), ('simplest', 'tasks'), ('tasks', 'feel'), ('feel', 'insurmountable'), ('insurmountable', 'im'), ('im', 'consumed'), ('consumed', 'fear'), ('fear', 'doubt'), ('doubt', 'every'), ('every', 'decision'), ('decision', 'feels'), ('feels', 'like'), ('like', 'minefield'), ('minefield', 'waiting'), ('waiting', 'explode'), ('explode', 'anxiety'), ('anxiety', 'grip'), ('grip', 'im'), ('im', 'powerless'), ('powerless', 'break'), ('break', 'free'), ('free', 'relentless'), ('relentless', 'hold'), ('hold', 'thoughts'), ('thoughts', 'race'), ('race', 'like'), ('like', 'runaway'), ('runaway', 'train'), ('train', 'cant'), ('cant', 'seem'), ('seem', 'find'), ('find', 'way'), ('way', 'slow'), ('slow', 'catch'), ('catch', 'breath'), ('breath', 'im'), ('im', 'trapped'), ('trapped', 'whirlwind'), ('whirlwind', 'fear'), ('fear', 'uncertainty'), ('uncertainty', 'cant'), ('cant', 'see

In [11]:
# ================================
# 11. NAMED ENTITY RECOGNITION (SpaCy)
# ================================
nlp = spacy.load("en_core_web_sm")

def extract_entities(text):
    doc = nlp(text)
    return [(ent.text, ent.label_) for ent in doc.ents]

df["entities"] = df["clean_text"].apply(extract_entities)

print("Sample NER:")
print(df["entities"].iloc[0])



Sample NER:
[]


In [11]:
# ================================
# 12. PREPARE TEXT FOR MODEL
# (Use lemmatized or stemmed text)
# ================================
df["final_text"] = df["lemmatized"].apply(lambda x: " ".join(x))

In [12]:
# ================================
# 13. TF-IDF (Unigram + Bigram)
# ================================
vectorizer = TfidfVectorizer(ngram_range=(1,2))  # Unigram + Bigram

X = vectorizer.fit_transform(df["final_text"])
y = df["status"]


In [13]:
# ================================
# 14. TRAIN TEST SPLIT
# ================================
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)



In [14]:
# ================================
# 15. MODEL TRAINING
# ================================
model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

LogisticRegression(max_iter=1000)

In [15]:
# ================================
# 16. EVALUATION
# ================================
y_pred = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n")
print(classification_report(y_test, y_pred))

Accuracy: 0.862063967533095

Classification Report:

                      precision    recall  f1-score   support

             anxiety       0.94      0.94      0.94      3492
             bipolar       0.98      0.92      0.95      2764
          depression       0.69      0.75      0.72      3186
              normal       0.78      0.96      0.86      3223
personality disorder       0.99      0.95      0.97      2823
              stress       0.97      0.87      0.92      2963
            suicidal       0.71      0.59      0.64      2247

            accuracy                           0.86     20698
           macro avg       0.87      0.85      0.86     20698
        weighted avg       0.87      0.86      0.86     20698



In [16]:
import joblib
joblib.dump(model,"mentalh_model.pkl")
joblib.dump(vectorizer,"vectorizr.pkl")

['vectorizr.pkl']

In [17]:
import os 
print(os.getcwd())

C:\Users\AsusROG1\Goonj\Mental Health-NLP
